# Data Audit

Prepare the repo inside Kaggle and inspect the locally generated Gemini-backed prompt inventory and split manifests. Data generation and augmentation are expected to happen before the repo is cloned into Kaggle.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None


def read_secret(name: str) -> str:
    if UserSecretsClient is not None:
        try:
            return UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = read_secret("FRS_REPO_URL")
HF_TOKEN = read_secret("HF_TOKEN")
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in Kaggle Secrets before running this notebook.")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

Cloning into '/kaggle/working/false-refusal-suppression'...


Obtaining file:///kaggle/working/false-refusal-suppression
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.

In [2]:
PILOT_PROMPTS = Path("data/processed/prompts/pilot_prompts_gemini.jsonl")
split_dir = Path("data/processed/splits/pilot_gemini")

required_paths = [
    PILOT_PROMPTS,
    split_dir / "discovery.jsonl",
    split_dir / "selection.jsonl",
    split_dir / "holdout.jsonl",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt local data artifacts. Generate them locally before running Kaggle: " + ", ".join(missing)
    )

print(PILOT_PROMPTS)
print(split_dir)

data/processed/prompts/pilot_prompts_gemini.jsonl
data/processed/splits/pilot_gemini


In [3]:
import json
import pandas as pd

records = []
with open(PILOT_PROMPTS, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
display(df.groupby(["group", "topic"]).size().reset_index(name="count"))
display(df.groupby("family_id").size().reset_index(name="examples_per_family"))
display(
    pd.DataFrame(
        {
            "split": ["discovery", "selection", "holdout"],
            "path": [
                str(split_dir / "discovery.jsonl"),
                str(split_dir / "selection.jsonl"),
                str(split_dir / "holdout.jsonl"),
            ],
        }
    )
)

,group,topic,count
0,benign_borderline,cloud_security_operations,8
1,benign_borderline,defensive_security_testing,8
2,benign_borderline,llm_security,8
3,benign_borderline,shell_process_management,8
4,benign_easy,git_basics,8
5,benign_easy,networking_basics,8
6,benign_easy,observability,8
7,benign_easy,python_basics,8
8,capability_holdout,algorithms,8
9,capability_holdout,containers,8


,family_id,examples_per_family
0,borderline_key_rotation,8
1,borderline_kill_process,8
2,borderline_prompt_injection,8
3,borderline_sql_injection_demo,8
4,easy_dns,8
5,easy_git_status,8
6,easy_logging,8
7,easy_python_dict,8
8,holdout_binary_search,8
9,holdout_docker,8


,split,path
0,discovery,data/processed/splits/pilot_gemini/discovery.j...
1,selection,data/processed/splits/pilot_gemini/selection.j...
2,holdout,data/processed/splits/pilot_gemini/holdout.jsonl
